## Hybrid Retriever- Combining Dense And Sparse Retriever

BLOG FOR BETTER UNDERSTNDING :
https://towardsdatascience.com/dance-between-dense-and-sparse-embeddings-enabling-hybrid-search-in-langchain-milvus-7c8de54dda24/



Youtube: https://www.youtube.com/watch?v=CK0ExcCWDP4

In [1]:
from langchain.schema import Document

# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

docs

c:\Users\heman\Desktop\05 Ultimate RAG Bootcamp Using Langchain,LangGraph and Langsmith by Krish Naik\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={}, page_content='LangChain helps build LLM applications.'),
 Document(metadata={}, page_content='Pinecone is a vector database for semantic search.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='Langchain can be used to develop agentic ai application.'),
 Document(metadata={}, page_content='Langchain has many types of retrievers.')]

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = OpenAIEmbeddings()

# Create FAISS vectorstore
dense_vectorstore = FAISS.from_documents(docs, embedding_model)

# Create Dense Retriever
dense_retriever = dense_vectorstore.as_retriever()

In [3]:
from langchain_community.retrievers import BM25Retriever

# Step 3: Sparse Retriever(BM25)
sparse_retriever = BM25Retriever.from_documents(docs)

sparse_retriever.k = 3 #top- k documents to retriever

In [4]:
from langchain.retrievers import EnsembleRetriever

# step 4: combine dense and sparse retriever using EnsembleRetriever = hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)

hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000021EBFD4D010>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000021EBFD4D400>, k=3)], weights=[0.7, 0.3])

In [5]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


### RAG Pipeline with hybrid retriever

In [6]:
from langchain.prompts import PromptTemplate

# Step 5: Prompt Template
prompt = PromptTemplate.from_template(
"""
Answer the question based on the context below.
Context: {context}
Question: {input}
"""
)

In [7]:
from langchain_openai import ChatOpenAI

# step 6-llm
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.2)
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000021EBFD4FCB0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021EBF4D8440>, root_client=<openai.OpenAI object at 0x0000021F09ED8410>, root_async_client=<openai.AsyncOpenAI object at 0x0000021F09ED8A50>, temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [8]:
from langchain.chains.combine_documents import create_stuff_documents_chain

#Step 7: Create stuff Docuemnt Chain
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\nContext: {context}\nQuestion: {input}\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000021EBFD4FCB0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021EBF4D8440>, root_client=<openai.OpenAI object at 0x0000021F09ED8410>, root_async_client=<openai.AsyncOpenAI object at 0x0000021F09ED8A50>, temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [9]:
from langchain.chains.retrieval import create_retrieval_chain

# Step 8: Create Retrieval Chain
rag_chain = create_retrieval_chain(
    retriever=hybrid_retriever,
    combine_docs_chain=document_chain
)

rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000021EBFD4D010>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000021EBFD4D400>, k=3)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\nContext: {context}\nQuestion: {input}\n')
            | ChatOpenAI(client=<openai.re

In [10]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: Output
print("✅ Answer:\n", response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer:
 You can build an app using LLMs by utilizing LangChain, which helps in developing LLM applications. LangChain can be used to develop agentic AI applications, and it offers many types of retrievers to enhance the functionality of your app. Additionally, you can also consider using Pinecone, a vector database for semantic search, to further optimize your app's performance.

📄 Source Documents:

Doc 1: LangChain helps build LLM applications.

Doc 2: Langchain can be used to develop agentic ai application.

Doc 3: Langchain has many types of retrievers.

Doc 4: Pinecone is a vector database for semantic search.
